## Parameter defintions

#### Load in required packages and JMMenura

In [ ]:
using Pkg
Pkg.activate(".")

# Set to location of JMMenura
include(#"./../../../JMMenura/src/JMMenura.jl")

# Import packages
using .JMMenura
using Phylo, Distributions, Random, JLD2, LinearAlgebra, PosDefManifold, Distances, StatsBase, GpABC


### Set up simulation

These parameters are for setting up an OU model.

#### Overall parameters

In [ ]:
# Number of traits to simulate
n = # SET

# Open tree - Set file path to tree
tree1 = open(parsenewick, #"./../..//anoles_data//bigsim.tre")

The tree is loaded in as a phylo object. Take note of the timescale to tree is on as this will affect the scale for the timestep `dt`. Recommended to scale the tree to be on the timscale 0 to 1. 

In [ ]:
# Find the root node of the trait
root = getroot(tree1)
root_num = tree1.nodedict[root.name]

Currently need to know the root of the tree to specify the parameters. This is recorded throught the node number of the root

#### Trait parameters

In [ ]:
# Set trait parameters needed to evolve traits
# Should be a vector of length n 
trait_alpha = #repeat([0.0], n)
trait_mu = #repeat([0.0], n)
trait_sigma = #repeat([sqrt(2)], n)

# Parameters used for reference simulation
trait_parameters_true = Dict(root_num => (alpha = trait_alpha, mu = trait_mu, sigma = trait_sigma))

Define trait parameters to simulate the OU process. Should be vectors of length `n` the number of traits to simulate.

* alpha - Mean reversion term (0 for brownian motion)
* mu - Mean for each trait
* sigma - Variability for each trait

The dictionary indexes which parameters are applied to which nodes. Parameters are applied to all downstream branches. Different paramaters can be specified for different nodes in which case they will applied to all descendant nodes. Parameters are applied starting from the root node descending through time.

#### Matrix parameters

In [ ]:
# Define G matrix - Can be done beforehand and then loaded in using JLD2
@load #"./../../anoles_data/P0.jld2"

# Variables needed for OU matrix model
mat_alpha = #0
mat_sigma = #sqrt(2)
mat_mu = #copy(P0)

# Create matrix dictionary
mat_parameters_true = Dict(root_num => (alpha = mat_alpha, mu = mat_mu, sigma = mat_sigma))

Define matrix parameters to simulate the OU process. Parameters are single values with the exception of the mu parameter.

* alpha - Mean reversion term (0 for brownian motion)
* mu - Mean for the G-matrix. Parameter is a covariance matrix of the same size which is the mean.
* sigma - Variability for the matrix

The dictionary indexes which parameters are applied to which nodes. Parameters are applied to all downstream branches. Different paramaters can be specified for different nodes in which case they will applied to all descendant nodes. Parameters are applied starting from the root node descending through time.

#### Simulation functions

In [ ]:
# Specify what evolution functions are used
# Can set dt the evolution lenght
trait_evol_func = trait_evol(#dt = 0.005)
mat_evol_func = mat_evol_affine(#dt = 0.005)

`dt` determines the timestep used when solving the SDEs. This is the most common parameter to change. The output of the above are functions which can be used by the SDE solver. 

Can also instead use the isospectral model by using `mat_evol_isospectral` to evolve the matrix. Note this will change the parameters needed to be `a` and `b` which are single numbers for the matrix.

#### Running simulation to create reference data

In [ ]:
ref_sim = menura_parameter_descend!(mat_parameters_true, trait_parameters_true, tree1, trait_evol_func
, mat_evol_func, 0.0, trait_mu, P0, true)

Definition for `menura_parameter_descend!` is 

`menura_parameter_descend!(mat_parameters, trait_parameters, tree, trait_evol, matrix_evol, t0, trait0, mat0, each)`

* `mat_parameters` - These are the parameters used to evolve the matrix
* `trait_parameters` - Similarly these are the parameters used to evolve the traits
* `tree` - This is the tree which the simulation occurs over. Note that this function will set up the tree.
* `trait_evol` - The function used to evolve the traits
* `matrix_evol` - The function used to evolve the matrix
* `t0` - Initial time to start simulation from
* `trait0` - The initial value the traits are set to at the root node
* `mat0` - The initial value the matrix is set to at the root node
* `each` - `true` if the G-matrix should be updated at each timestep.

Output is a tuple of a tree and a bool which states if the simulation remained stable.

#### Saving the reference data


In [ ]:
ref_data = get_data(ref_sim)
@save "ref_data.jld2" ref_data
@save "tree.jld2" tree1

`get_data` extracts the data in a usable format for GpABC from the above output.

## JMMABCparameters

JMMABCparameters are a group of structs which defined the paramaters, models and assumptions of JMMenura ABC simulations. They are used for both model and parameter selection. Important examples are:

### `JMMABCAlphaDifferentConstant` 

Assumptions:
* OU model for matrix and traits
* Only traits being evolved are alpha for matrix and traits
* Each trait has a different alpha value 
* Parameters remain constant down the entire tree. 

`JMMABCAlphaDifferentConstant(trait_alpha_prior, trait_mu_known, trait_sigma_known, mat_alpha_prior, mat_mu_known, mat_sigma_known, n)`

Inputs of this type are

* `trait_alpha_prior` - Array of length `n` which contains the prior for each of the different traits
* `trait_mu_known` - Known mean for the traits which is an array of length `n`.
* `trait_sigma_known` - Array of length `n` which contains the sigma value for each trait
* `mat_alpha_prior` - Single prior for the matrix alpha
* `mat_mu_known` - `n` by `n` matrix which is the mean of the matrix
* `mat_sigma_known` - Known sigma value for the matrix
* `n` - Integer stating the number of traits



### `JMMABCAlphaEqualConstant` 

Assumptions:
* OU model for matrix and traits
* Only traits being evolved are alpha for matrix and traits
* All traits share an alpha value
* Parameters remain constant down the entire tree. 

`JMMABCAlphaEqualConstant(trait_alpha_prior, trait_mu_known, trait_sigma_known, mat_alpha_prior, mat_mu_known, mat_sigma_known, n)`

Inputs of this type are

* `trait_alpha_prior` - Single prior for all traits alpha
* `trait_mu_known` - Known mean for the traits which is an array of length `n`.
* `trait_sigma_known` - Array of length `n` which contains the sigma value for each trait
* `mat_alpha_prior` - Single prior for the matrix alpha
* `mat_mu_known` - `n` by `n` matrix which is the mean of the matrix
* `mat_sigma_known` - Known sigma value for the matrix
* `n` - Integer stating the number of traits


### `JMMABCBrownian` 

Assumptions:
* OU model for traits and BM model for matrix
* Only traits being evolved are alpha for traits and sigma for matrix
* All traits share a sigma value
* Parameters remain constant down the entire tree. 

`JMMABCBrownian(trait_alpha_prior, trait_mu_known, trait_sigma_known, mat_mu_known, mat_sigma_prior, n)`

Inputs of this type are

* `trait_alpha_prior` - Single prior for all traits alpha
* `trait_mu_known` - Known mean for the traits which is an array of length `n`.
* `trait_sigma_known` - Array of length `n` which contains the sigma value for each trait
* `mat_mu_known` - `n` by `n` matrix which is the mean of the matrix
* `mat_sigma_prior` - Single prior for the sigma value of the matrix
* `n` - Integer stating the number of traits


### `JMMABCIsospectralAlphaAB` 

Assumptions:
* OU model for traits and isospectral model for matrix
* Only traits being evolved are alpha for traits and a and b parameters for matrix
* All traits share a sigma value
* Parameters remain constant down the entire tree. 

`JMMABCIsospectralAlphaAB(trait_alpha_prior, trait_mu_known, trait_sigma_known, Iso_prior, Iso_prior, n)`

Inputs of this type are

* `trait_alpha_prior` - Single prior for all traits alpha
* `trait_mu_known` - Known mean for the traits which is an array of length `n`.
* `trait_sigma_known` - Array of length `n` which contains the sigma value for each trait
* `mat_a_prior` - Single prior for the a parameter of the matrix
* `mat_b_prior` - Single prior for the b parameter of the matrix
* `n` - Integer stating the number of traits

## Parallelisation

To help speed up the task the threshold is precalculated. GpABC is then performed by several identical tasks each seeking a proportion of the desired parameters to sample. Each job has a threshold and a run file with the same code until after the prior definition. The other difference is that if reference data has to be created this is handled by the threshold file with the run file just reading in the reference data using JLD2.


## Parameter selection

#### Prior definition

In [ ]:
prior = Gamma(2, 0.25)
para = JMMABCAlphaDifferentConstant([prior for _ in 1:n], trait_mu, trait_sigma, prior, mat_mu, mat_sigma, n)

#### Creating threshold

In [ ]:
thresholds = test_threshold(ref_data, tree1, para, zeros(n), P0, 1000, dt = 0.005, each = true)
threshold = sort(thresholds)[4]

`test_threshold` has the definition

`test_threshold(reference_data, tree, JMMpara, trait0, mat0, n_particles; t0 = 0.0, each = false, dt = 0.001, distance_function = trait_mat_distance(JMMpara.size,nleaves(tree)), summary_function = get_data, verbose = true)`

Inputs are 

* `reference_data` - Reference data formated in the way given by `get_data`
* `tree` - The tree used for evolution
* `JMMpara` - JMMABCparameters object used
* `trait0` - The initial value the traits are set to at the root node
* `mat0` - The initial value the matrix is set to at the root node
* `n_particles` - The number of points to generate a threshold for

Optional inputs

* `t0` - Start time of simulation
* `each` - `true` if the G-matrix should be updated at each timestep.
* `dt` - Step size used by SDE solver
* `distance_function` - The function used to define distance. Included to easily allow changing weightings, if traits not measured etc.
* `summary_function` - Specifies how the data from the simulation is summarised. Allows for traits or matrix evolution to not be counted
* `verbose` - If `true` outputs all warnings about instability

Once thresholds are generated can choose a value to approximate selecting a certain proportion of points

#### Save threshold

In [ ]:
# Save threshold
@save "./threshold.jld2" threshold
s_para_result = Vector{Any}()
@save "./s_para_result.jld2" s_para_result

To allow for the process to be parallelised the threshold is precalculated and saved for future use. Also creates an empty vector to save the results to.

#### Sampling points

In [ ]:
n_particles = 100
@load "./threshold.jld2" threshold

run_result = menura_bayesian(para_ref_data, tree1, para, zeros(n), P0, threshold, n_particles
, dt = 0.005, max_iter = 50000*n_particles, each = true)

The number of particles to sample is selected. The function `menura_bayesian` is used to perform GpABC. The function has definition

`menura_bayesian(reference_data, tree, JMMpara, trait0, mat0, threshold, n_particles; max_iter = 50*n_particles, t0 = 0.0, each = false, dt = 0.001, distance_function = trait_mat_distance(JMMpara.size,nleaves(tree)), summary_function = get_data, verbose = true)`

and inputs 

* `reference_data` - Reference data formated in the way given by `get_data`
* `tree` - The tree used for evolution
* `JMMpara` - JMMABCparameters object used
* `trait0` - The initial value the traits are set to at the root node
* `mat0` - The initial value the matrix is set to at the root node
* `n_particles` - The number of points to retain

Optional inputs

* `max_iter` - The maximum number of iterations which will be run seeking `n_particles`. If the maximum is reached the simulation is stopped and the found particles are returned.
* `t0` - Start time of simulation
* `each` - `true` if the G-matrix should be updated at each timestep.
* `dt` - Step size used by SDE solver
* `distance_function` - The function used to define distance. Included to easily allow changing weightings, if traits not measured etc.
* `summary_function` - Specifies how the data from the simulation is summarised. Allows for traits or matrix evolution to not be counted
* `verbose` - If `true` outputs all warnings about instability

#### Saving sampled points

In [ ]:
# Save model results
@load "./s_para_result.jld2" s_para_result
push!(s_para_result, run_result)
@save "s_para_result.jld2" s_para_result

Pushes the output GpABC object in the vector storing the simulation results. Important the vector storing the result is loaded **AFTER** the simulation is run as if it is loaded before the will the parallel processes will override each other.

## Model selection

The creation of reference data is the same for model selection runs until the definition of priors.

#### Prior definitions

In [ ]:
OU_prior = Gamma(2, 0.25)
OU_para = JMMABCAlphaEqualConstant(OU_prior, trait_mu, trait_sigma, OU_prior, mat_mu, mat_sigma, n)

BW_prior = Gamma(2, 0.25)
BW_para = JMMABCBrownian(BW_prior, trait_mu, trait_sigma, mat_mu, BW_prior, n)

Iso_prior = Gamma(2, 0.25)
Iso_para = JMMABCIsospectralAlphaAB(Iso_prior, trait_mu, trait_sigma, Iso_prior, Iso_prior, n)

Similar to parameter selection the priors can be defined using JMMABCparameters. Different parameter types encode different model types and assumptions. See above for defintions and assumptions

#### Function defintions

In [ ]:
# Creates functions used to simulate different models
OU_func = create_bayesian_sim(tree1, OU_para, trait_mu, mat_mu, dt = 0.005, each = true)
BW_func = create_bayesian_sim(tree1, BW_para, trait_mu, mat_mu, dt = 0.005, each = true)
Iso_func = create_bayesian_sim(tree1, Iso_para, trait_mu, mat_mu, dt = 0.005, each = true)

model_sim_functions = [OU_func, BW_func, Iso_func]
priors = [get_priors(OU_para), get_priors(BW_para), get_priors(Iso_para)]

dist_func = trait_mat_distance(n, nleaves(tree1))

To use the GpABC model selection functions a vector of functions and a vector of priors is required. The function `create_bayesian_sim` handles the creation of the mentioned functions given a JMMABCparameters object (the object ensures the relevant assumptions are accounted for). The function has defintion 

`create_bayesian_sim(tree, JMMpara, trait0, mat0; t0 = 0.0, each = true, dt = 0.001, verbose = true, summary_function = false)`

inputs

* `tree` - The tree used for evolution
* `JMMpara` - JMMABCparameters object used
* `trait0` - The initial value the traits are set to at the root node
* `mat0` - The initial value the matrix is set to at the root node

and optional inputs 

* `t0` - Start time of simulation
* `each` - `true` if the G-matrix should be updated at each timestep.
* `dt` - Step size used by SDE solver
* `summary_function` - Specifies how the data from the simulation is summarised. Allows for traits or matrix evolution to not be counted
* `verbose` - If `true` outputs all warnings about instability


The function `get_prior` extracts all prior distributions from a JMMABCparameters object. The distance function must also be defined at this stage.

#### Thresholds

In [ ]:
# Set number of points to generate threshold for
n_threshold = #1000

# Determine threshold
OU_thresholds = [test_threshold(ref_data, tree1, OU_para, trait_mu, mat_mu, n_threshold, each = true),
                test_threshold(ref_data, tree1, BW_para, trait_mu, mat_mu, n_threshold, each = true), 
                test_threshold(ref_data, tree1, Iso_para, trait_mu, mat_mu, n_threshold, each = true)]
                
threshold = minimum([sort(model_thresholds)[4] for model_thresholds in OU_thresholds])

# Save threshold
@save "./threshold.jld2" threshold
s_model_result = Vector{Any}()
@save "./s_model_result.jld2" s_model_result


Much like for parameter selection the `test_threshold` function is used to generate the thresholds. Note that it must be run independently for each model to be tested and a single overall threshold must be decided.

#### Model selection

In [ ]:
# Set number of particles to accept and load threshold
n_particles = 200
@load "threshold.jld2"

# Simulate number of points
sim_out = SimulatedModelSelection(ref_data, model_sim_functions, priors, [threshold], n_particles, distance_function = dist_func,
max_iter = 20000*n_particles)

# Save model results
@load "./s_model_result.jld2"
push!(s_model_result, sim_out)
@save "s_model_result.jld2" s_model_result

With the models and thresholds defined the `SimulatedModelSelection` function from GpABC is used to perform model selection. Like above the result is saved in a JLD2 file.